In [ ]:
from pathlib import Path
import sys
import numpy as np
import yaml
from PIL import Image, ImageDraw

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.build import build_finetune_loader

config = yaml.safe_load((ROOT / 'config/finetune_test.yaml').read_text())
loader_config = dict(config['data']['train'])
loader_config['batch_size'] = 2 * 2
loader_config['num_workers'] = 0
loader = build_finetune_loader(loader_config, train=True)

def draw_item(batch, index):
    array = batch['image'][index].permute(1, 2, 0).numpy()
    image = Image.fromarray(np.clip((array + 1) * 127.5, 0, 255).astype('uint8'))
    target = batch['target'][index, 0].numpy()
    mask = Image.fromarray((target * 255).astype('uint8')).resize(image.size)
    color = Image.new('RGB', image.size, (255, 40, 160))
    image = Image.composite(Image.blend(image, color, 0.35), image, mask)
    draw = ImageDraw.Draw(image)
    prompt = batch['prompt'][index]
    if prompt['points'] is not None:
        for x, y in prompt['points']:
            draw.ellipse((x - 10, y - 10, x + 10, y + 10), fill='cyan')
    if prompt['box'] is not None:
        draw.rectangle(tuple(prompt['box']), outline='cyan', width=5)
    cond = int(batch['cond'][index])
    label = batch['label_target'][index].int().tolist()
    draw.text((10, 10), f"{prompt['type']}  cond={cond}  class={label}", fill='white')
    return image.resize((504, 504))


In [ ]:
batch = next(loader)
panels = [draw_item(batch, index) for index in range(4)]
sheet = Image.new('RGB', (1008, 1008), 'white')
for index, panel in enumerate(panels):
    sheet.paste(panel, ((index % 2) * 504, (index // 2) * 504))
sheet
